In [ ]:
from orion.utils.misc_utils import *
from orion.utils.o3d_utils import *
import numpy as np
import torch
import json
import h5py
import cv2
import sys
from PIL import Image
import PIL
dust3r_path = './third_party/dust3r'
sys.path.append(dust3r_path)
from dust3r.utils.geometry import geotrf
from tqdm import tqdm


In [ ]:
def draw_scene_3d(
                      pcd_colors,
                      pcd_points,
                      draw_trajs=False,
                      draw_keypoints=False,
                      additional_points=None,
                      downsample=True,
                      depth_trunc=3.0,
                      no_background=False,
                      overlay_annotation=False,
                      additional_point_draw_lines=False,
                      use_white_bg=False,
                      skip_trajs=[],
                      skip_keypoints=[],
                      offline=False,
):

        marker_size = 3
        if offline:
            fig = plotly_draw_3d_pcd(pcd_points, pcd_colors, addition_points=additional_points, marker_size=marker_size, no_background=no_background, additional_point_draw_lines=additional_point_draw_lines, offline=offline)
            return fig
        else:
            plotly_draw_3d_pcd(pcd_points, pcd_colors, addition_points=additional_points, marker_size=marker_size, no_background=no_background, additional_point_draw_lines=additional_point_draw_lines)
            return pcd_points, pcd_colors

In [ ]:
annotation_folder = "/home/arisrei/research/RPL/ORION/annotations/human_demo/boat_gen6d_still"
demo_name = annotation_folder.split("/")[-1]
with open(os.path.join(annotation_folder, "config.json"), "r") as f:
    config_dict = json.load(f)
img_indexes = np.array([0,30,60,90,126])
subsequence_range = [0, 25]


In [ ]:
demo_file = h5py.File(config_dict["original_file"], "r")

depths = demo_file["data/human_demo/obs/dust3r_depth"][()][subsequence_range[0]:subsequence_range[1]]
images = demo_file["data/human_demo/obs/dust3r_rgb"][()][subsequence_range[0]:subsequence_range[1]]


intrinsics = demo_file["data/human_demo/obs/dust3r_intrinsics"][()][subsequence_range[0]:subsequence_range[1]]
#extrinsics = demo_file["data/human_demo/obs/dust3r_extrinsics"][()]
extrinsics = demo_file["data/human_demo/obs/dust3r_extrinsics_correct_frame"][()][subsequence_range[0]:subsequence_range[1]]
cotracker_dict = torch.load(os.path.join(annotation_folder, "tracked_points.pt"), map_location=torch.device('cuda:3'))
trajs_2d = cotracker_dict['pred_tracks'].cpu().detach().numpy()[0]
trajs_2d = trajs_2d[subsequence_range[0]:subsequence_range[1]]

scene_points = demo_file["data/human_demo/obs/scene_points"][()]
scene_colors = demo_file["data/human_demo/obs/scene_col"][()]

masks_orig = np.load(os.path.join(annotation_folder, 'masks.npz'))
masks_orig = masks_orig['arr_0']
masks_orig = masks_orig[subsequence_range[0]:subsequence_range[1]]

extrinsic_batches = demo_file["data/human_demo/obs/dust3r_extrinsic_batches"][()]
intrinsic_batches = demo_file["data/human_demo/obs/dust3r_intrinsic_batches"][()]
transforms_per_batch = demo_file['data/human_demo/obs/dust3r_transforms_per_batch'][()]
depths_per_batch = demo_file['data/human_demo/obs/dust3r_depths_unormalized_per_batch'][()]
demo_file.close()



In [ ]:
print(depths.shape, images.shape, intrinsics.shape, extrinsics.shape, trajs_2d.shape, scene_points.shape, scene_colors.shape, masks_orig.shape)

#make sure that there are same number of elements in depths, images, intrinsics.
#extrinsics, trajs_2d
#assert depths.shape[0] == images.shape[0] == intrinsics.shape[0] == extrinsics.shape[0] == trajs_2d.shape[0] == masks_orig.shape[0]

In [ ]:
def create_video(frames, output_path, fps=30, greyscale=False):
    height, width = frames[0].shape[0:2]
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # You can also use 'XVID', 'MJPG', etc.
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    if not greyscale:
        frames = frames[:,:,:,::-1]


    for frame in frames:
        if greyscale:
            frame = cv2.cvtColor(frame, cv2.COLOR_GRAY2RGB)
        out.write(frame)

    out.release()

create_video((images * 255).astype(np.uint8), os.path.join(annotation_folder, "dust3r/video.mp4"))
create_video((depths * 255).astype(np.uint8), os.path.join(annotation_folder, "dust3r/depth_video.mp4"), greyscale=True)

In [ ]:
display(Image.fromarray((depths[0] * 255).astype(np.uint8)))

In [ ]:
idx = 0
img = (images[idx]).astype(np.uint8)
curr_traj = trajs_2d[idx]
for i in range(curr_traj.shape[0]):
    x = int(curr_traj[i, 0])
    y = int(curr_traj[i, 1])
    img = cv2.circle(img, (x, y), 3, (0, 0, 255), -1)
display(Image.fromarray(img))

In [ ]:
traj_idx = 0
img = (images[0]).astype(np.uint8)
for i in range(trajs_2d.shape[0]):
    curr_traj = trajs_2d[i]
    x = int(curr_traj[traj_idx, 0])
    y = int(curr_traj[traj_idx, 1])
    img = cv2.circle(img, (x, y), 3, (0, 0, 255), -1)
display(Image.fromarray(img))

In [ ]:
#filter out trajectories that have coordinates that are out of bounds
max_x, max_y = images[0].shape[1] - 1, images[0].shape[0] - 1

trajs_2d_filtered = []
valid_traj_indices = []
for traj in trajs_2d:
    valid_indices = np.all((traj >= 0) & (traj <= [max_x, max_y]), axis=1)
    trajs_2d_filtered.append(traj[valid_indices])
    valid_traj_indices.append(np.argwhere(valid_indices == True))

print(len(valid_traj_indices), valid_traj_indices[0][0:5])

In [ ]:
traj_2d_pts = [[] for _ in range(trajs_2d.shape[1])]
for i in range(len(trajs_2d_filtered)):
    curr_frame_pts = trajs_2d_filtered[i]
    for idx, point in enumerate(curr_frame_pts):
        traj_2d_pts[valid_traj_indices[i][idx].item()].append(point)

test = np.array(traj_2d_pts[0], dtype=np.int32)
img = (images[0]).astype(np.uint8)
for i in range(trajs_2d.shape[0]):
    img = cv2.circle(img, (test[i, 0], test[i, 1]), 3, (0, 0, 255), -1)
display(Image.fromarray(img))

In [ ]:
len(trajs_2d_filtered), depths.shape, intrinsics.shape

In [ ]:
traj_3d_pts = [[] for _ in range(trajs_2d.shape[1])]
for i in range(len(trajs_2d_filtered)):
    untransformed_trajs = create_point_clouds_from_keypoints(trajs_2d_filtered[i], depths[i], intrinsics[i])
    untransformed_trajs *= 1000
    transformed_trajs = geotrf(extrinsics[i], untransformed_trajs)
    assert untransformed_trajs.shape == transformed_trajs.shape
    for idx, point in enumerate(transformed_trajs):
        traj_3d_pts[valid_traj_indices[i][idx].item()].append(point)
    #traj_3d_pts.append(transformed_trajs)

traj_3d_pts = [np.array(traj) for traj in traj_3d_pts]
print(len(traj_3d_pts))
print([traj_3d_pts[x].shape for x in range(5)])


In [ ]:
dust3r_mask = np.zeros((images.shape[0], images.shape[1], images.shape[2]), dtype=np.uint8)
for i in range(images.shape[0]):
    dust3r_mask[i] = np.array(Image.fromarray(masks_orig[i]).resize((images.shape[2], images.shape[1]), resample=PIL.Image.NEAREST))
display(Image.fromarray(dust3r_mask[0] * 100))

i = 0
target_ob_mask = 1
dust3r_mask[dust3r_mask != target_ob_mask] = 0 
masked = images[i] * 255 * dust3r_mask[i][:,:,np.newaxis]
display(Image.fromarray((images[i] * 255).astype(np.uint8)))
display(Image.fromarray(masked.astype(np.uint8)))    
display(Image.fromarray(dust3r_mask[i] * 255))

In [ ]:
# for i in range(img_indexes.shape[0]):
#     print(extrinsics[img_indexes[i]])

#make sure there no duplicate images
for i in range(images.shape[0]):
    for j in range(i + 1, images.shape[0]):
        assert not np.all(images[i] == images[j])
        assert not np.all(extrinsics[i] == extrinsics[j])

In [ ]:
img_indexes = np.array([0, 10, 24])

obj_pts = []
obj_cols = []
all_object_pts = []
all_object_colors = []

for q in tqdm(range(img_indexes.shape[0])):
    keypoints = []
    colors = []
    trunc = 1000
    print(images[q].shape)

    for i in range(images[img_indexes[q]].shape[0]):
        for j in range(images[img_indexes[q]].shape[1]):
            if dust3r_mask[img_indexes[q]][i, j] == 1:
                keypoints.append(np.asarray([j, i]))
                colors.append(images[img_indexes[q]][i, j])

    keypoints = np.asarray(keypoints)
    colors = np.asarray(colors)
    points_3d = create_point_clouds_from_keypoints(keypoints, depths[img_indexes[q]], intrinsics[img_indexes[q]])
    #points_3d = create_point_clouds_from_keypoints(keypoints, depths[q],
    #intrinsics[q])
    points_3d *= 1000
    #points_3d = geotrf(extrinsics[q], points_3d)

    #visualize in open3d
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points_3d)
    pcd.colors = o3d.utility.Vector3dVector(colors)
    pcd.transform(extrinsics[img_indexes[q]])
    task_relevant_obj_only = O3DPointCloud()
    task_relevant_obj_only.pcd = pcd
    task_relevant_obj_only.max_points = 40000
    obj_pts.append(np.array(pcd.points))
    curr_cols = np.array(pcd.colors)
    #curr_cols[:,:] = np.array([255, 255, 255]) / (q + 1)
    obj_cols.append(curr_cols)

    fig = visualize_o3d_point_cloud(task_relevant_obj_only, offline=True)
    display(fig)

all_object_colors = np.concatenate(obj_cols, axis=0)
all_object_pts = np.concatenate(obj_pts, axis=0)


In [ ]:
concatenated_traj_3d_pts = []
for i in range(len(traj_3d_pts)):
    if len(traj_3d_pts[i]) != 0:
        concatenated_traj_3d_pts.append(traj_3d_pts[i])
traj_3d_no_empty_pts = concatenated_traj_3d_pts
concatenated_traj_3d_pts = np.concatenate(concatenated_traj_3d_pts, axis=0)

In [ ]:
traj_3d_col = np.array([255, 0, 0]) * np.ones((concatenated_traj_3d_pts.shape[0], 3))

trajs_pcd = o3d.geometry.PointCloud()
# trajs_pcd.points = o3d.utility.Vector3dVector(traj_3d_pts[0])
# trajs_pcd.colors = o3d.utility.Vector3dVector(traj_3d_col[0:traj_3d_pts[0].shape[0]])
trajs_pcd.points = o3d.utility.Vector3dVector(concatenated_traj_3d_pts)
trajs_pcd.colors = o3d.utility.Vector3dVector(traj_3d_col)

pcd_wrapped = O3DPointCloud()
pcd_wrapped.pcd = trajs_pcd
pcd_wrapped.max_points = 40000
print(np.array(pcd_wrapped.pcd.points).shape, np.array(pcd_wrapped.pcd.colors).shape)
#pcd_wrapped.preprocess()
print(np.array(pcd_wrapped.pcd.points).shape, np.array(pcd_wrapped.pcd.colors).shape)

fig = visualize_o3d_point_cloud(pcd_wrapped, offline=True)
display(fig)

In [ ]:
scene_pcd = o3d.geometry.PointCloud()
scene_pcd.points = o3d.utility.Vector3dVector(scene_points)
scene_pcd.colors = o3d.utility.Vector3dVector(scene_colors)

pcd_wrapped = O3DPointCloud()
pcd_wrapped.pcd = scene_pcd
pcd_wrapped.max_points = 40000
print(np.array(pcd_wrapped.pcd.points).shape, np.array(pcd_wrapped.pcd.colors).shape)
pcd_wrapped.preprocess()
print(np.array(pcd_wrapped.pcd.points).shape, np.array(pcd_wrapped.pcd.colors).shape)

fig = visualize_o3d_point_cloud(pcd_wrapped, offline=True)
display(fig)

In [ ]:
objs_pcd = o3d.geometry.PointCloud()
objs_pcd.points = o3d.utility.Vector3dVector(all_object_pts)
objs_pcd.colors = o3d.utility.Vector3dVector(all_object_colors)

pcd_wrapped = O3DPointCloud()
pcd_wrapped.pcd = scene_pcd
pcd_wrapped.max_points = 40000
print(np.array(pcd_wrapped.pcd.points).shape, np.array(pcd_wrapped.pcd.colors).shape)
pcd_wrapped.preprocess()
pcd_wrapped.pcd += objs_pcd
print(np.array(pcd_wrapped.pcd.points).shape, np.array(pcd_wrapped.pcd.colors).shape)

fig = visualize_o3d_point_cloud(pcd_wrapped, offline=True)
display(fig)

In [ ]:
scene_pcd = o3d.geometry.PointCloud()
scene_pcd.points = o3d.utility.Vector3dVector(scene_points)
scene_pcd.colors = o3d.utility.Vector3dVector(scene_colors)

pcd_wrapped = O3DPointCloud()
pcd_wrapped.pcd = scene_pcd
pcd_wrapped.max_points = 40000
print(np.array(pcd_wrapped.pcd.points).shape, np.array(pcd_wrapped.pcd.colors).shape)
pcd_wrapped.preprocess()
pcd_wrapped.pcd = pcd_wrapped.pcd + trajs_pcd
print(np.array(pcd_wrapped.pcd.points).shape, np.array(pcd_wrapped.pcd.colors).shape)

fig = visualize_o3d_point_cloud(pcd_wrapped, offline=True)
display(fig)

In [ ]:
# scene_pcd = o3d.geometry.PointCloud()
# scene_pcd.points = o3d.utility.Vector3dVector(scene_points)
# scene_pcd.colors = o3d.utility.Vector3dVector(scene_colors)

# pcd_wrapped = O3DPointCloud()
# pcd_wrapped.pcd = scene_pcd + trajs_pcd + all_objs_pcd
# pcd_wrapped.max_points = 40000
# print(np.array(pcd_wrapped.pcd.points).shape, np.array(pcd_wrapped.pcd.colors).shape)
# pcd_wrapped.preprocess()
# print(np.array(pcd_wrapped.pcd.points).shape, np.array(pcd_wrapped.pcd.colors).shape)

# fig = visualize_o3d_point_cloud(pcd_wrapped)
# display(fig)
# fig.write_html(os.path.join(annotation_folder, "scene_and_object_pcds_extrinsic_transform_from_overlapping_inputs.html"))

In [ ]:
print(len(traj_3d_no_empty_pts))
print(traj_3d_no_empty_pts[0].shape)

In [ ]:
scene_pcd = o3d.geometry.PointCloud()
scene_pcd.points = o3d.utility.Vector3dVector(scene_points)
scene_pcd.colors = o3d.utility.Vector3dVector(scene_colors)

pcd_wrapped = O3DPointCloud()
pcd_wrapped.pcd = scene_pcd + objs_pcd
pcd_wrapped.max_points = 40000
print(np.array(pcd_wrapped.pcd.points).shape, np.array(pcd_wrapped.pcd.colors).shape)
pcd_wrapped.preprocess()
print(np.array(pcd_wrapped.pcd.points).shape, np.array(pcd_wrapped.pcd.colors).shape)

plane_estimation_kwargs={
        "ransac_n": 3,
        "num_iterations": 1000,
        "distance_threshold": 0.01
    }

plane_estimation_result = pcd_wrapped.plane_estimation(**plane_estimation_kwargs)
T_xy_plane_align = estimate_rotation(plane_estimation_result["plane_model"], z_up=False)
pcd_wrapped.pcd.transform(T_xy_plane_align)

transformed_traj = []
for i in range(len(traj_3d_no_empty_pts)):
        segment = traj_3d_no_empty_pts[i]
        #segment = simple_filter_outliers(segment)
        segment = transform_points(T_xy_plane_align, segment)
        transformed_traj.append(segment)

#fig = draw_scene_3d(np.array(pcd_wrapped.pcd.colors), np.array(pcd_wrapped.pcd.points), offline=True, downsample=True)
fig = draw_scene_3d(np.array(pcd_wrapped.pcd.colors), np.array(pcd_wrapped.pcd.points), draw_trajs=False, additional_points=transformed_traj, additional_point_draw_lines=True, offline=True, downsample=True)
display(fig)

In [ ]:
fr_0_obj_pcd = o3d.geometry.PointCloud()
fr_0_obj_pcd.points = o3d.utility.Vector3dVector(obj_pts[0])
fr_0_obj_pcd.colors = o3d.utility.Vector3dVector(obj_cols[0])

pcd_wrapped = O3DPointCloud()
pcd_wrapped.pcd = fr_0_obj_pcd
pcd_wrapped.max_points = 40000
print(np.array(pcd_wrapped.pcd.points).shape, np.array(pcd_wrapped.pcd.colors).shape)
pcd_wrapped.preprocess()
pcd_wrapped.pcd = pcd_wrapped.pcd + trajs_pcd
print(np.array(pcd_wrapped.pcd.points).shape, np.array(pcd_wrapped.pcd.colors).shape)

fig = draw_scene_3d(np.array(pcd_wrapped.pcd.colors), np.array(pcd_wrapped.pcd.points), draw_trajs=True, additional_points=traj_3d_no_empty_pts, additional_point_draw_lines=True, offline=True, downsample=True)
display(fig)

In [ ]:
fig.write_html(os.path.join(annotation_folder, "frame_0_obj_pcd_and_trajs.html"))